In [37]:
import numpy as np
import math
import time
import copy
import random

In [38]:
kolumny_duże = 3
wiersze_duże = 3
kolumny_małe = 3
wiersze_małe = 3
koniec = None
remis = False
pozycja = False
ZNAK_GRACZ = str()
tura = str()
następna_plansza = None # współrzędne (k, w) małej planszy, na której trzeba grać

def tworzenie_planszy():
  # 9 plansz 3x3 # plansze[wiersze_duże][kolumny_duże][wiersze_małe][kolumny_małe]
  plansza = [[[[" " for _ in range(3)] for _ in range(3)] for _ in range(3)] for _ in range(3)]
  # 3x3 do śledzenia wygranych w małych kwadratach
  duża_plansza = [[" " for _ in range(3)] for _ in range(3)]

  return plansza, duża_plansza

def wyświetlenie_planszy(plansza):
  for w_duży in range(3):
    print("-" * 25)
    for w_mały in range(3):
      line = ""
      for k_duży in range(3):
        line += "| " + " ".join(plansza[w_duży][k_duży][w_mały]) + " "
      print(line + "|")
  print("-" * 25)


In [39]:
def sprawdź_wygrana_3x3(sektor):
  for wiersz in range(3):
    if sektor[wiersz][0] == sektor[wiersz][1] == sektor[wiersz][2] != " ":
      return sektor[wiersz][0] # zwraca "X" lub "O"

  for kolumna in range(3):
    if sektor[0][kolumna] == sektor[1][kolumna] == sektor[2][kolumna] != " ":
      return sektor[0][kolumna]

  if sektor[0][0] == sektor[1][1] == sektor[2][2] != " ":
    return sektor[0][0]
  if sektor[0][2] == sektor[1][1] == sektor[2][0] != " ":
    return sektor[0][2]

  return None # nikt nie wygrał

def czy_remis(sektor):
  if sprawdź_wygrana_3x3(sektor) is not None:
    return False
  return all(pole != " " for wiersz in sektor for pole in wiersz)

In [40]:
def stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k):
  mała_p = plansza[pozycja_d_w][pozycja_d_k]
  zwycięzca = sprawdź_wygrana_3x3(mała_p)

  if zwycięzca != None:
    duża_plansza[pozycja_d_w][pozycja_d_k] = zwycięzca
    return True # sektor wypełniony

  if czy_remis(mała_p) == True:
    duża_plansza[pozycja_d_w][pozycja_d_k] = "R"
    return True

  return False # na sektorze można grać

In [41]:
def wypełnienie(plansza, duża_plansza):
  for d_w in range(3):
    for d_k in range(3):
      stan = duża_plansza[d_w][d_k]
      if stan != " " and stan != "R": # jeśli na małej planszy wygrał O / X
        for m_w in range(3):
          for m_k in range(3):
            plansza[d_w][d_k][m_w][m_k] = stan
      elif stan == "R":
        for m_w in range(3):
          for m_k in range(3):
            plansza[d_w][d_k][m_w][m_k] = "-"

MINMAX

In [42]:
def heurystyka(plansza, duża_p):
  wynik = 0

  # 1. Punkty za wygranie małej planszy
  for d_w in range(3):
    for d_k in range(3):
      if duża_p[d_w][d_k] == ZNAK_GRACZ: wynik += 100
      elif duża_p[d_w][d_k] == ZNAK_KOMPUTER: wynik -= 100

  linie = [[(0,0), (0,1), (0,2)], [(1,0), (1,1), (1,2)], [(2,0), (2,1), (2,2)], # wiersze
      [(0,0), (1,0), (2,0)], [(0,1), (1,1), (2,1)], [(0,2), (1,2), (2,2)], # kolumny
      [(0,0), (1,1), (2,2)], [(0,2), (1,1), (2,0)]] # przekątne

  for linia in linie:
    symbole = [duża_p[w][k] for w, k in linia]

    # pomijanie linii, gdzie jest remis, bo tam już i tak nikt nie wygra
    if "R" in symbole:
      continue

    # 2. Punkty za wygranie dwóch małych plansz w linii
    # dwa pola komputera i jeden pusty (bez blokady przez gracza)
    if symbole.count(ZNAK_GRACZ) == 2 and symbole.count(" ") == 1: wynik += 200
    if symbole.count(ZNAK_KOMPUTER) == 2 and symbole.count(" ") == 1: wynik -= 200

    # 3. Punkty za blokowanie wygranej przeciwnika
    if symbole.count(ZNAK_KOMPUTER) == 2 and symbole.count(ZNAK_GRACZ) == 1: wynik += 150
    if symbole.count(ZNAK_GRACZ) == 2 and symbole.count(ZNAK_KOMPUTER) == 1: wynik -= 150

  for d_w in range(3):
    for d_k in range(3):
      if duża_p[d_w][d_k] == " ":
        mała_plansza = plansza[d_w][d_k]
        for linia in linie:
          symbole_małe = [mała_plansza[m_w][m_k] for m_w, m_k in linia]
          # 4. Punkty za dwa znaki w linii na małej planszy
          if symbole_małe.count(ZNAK_GRACZ) == 2 and symbole_małe.count(" ") == 1: wynik += 5
          if symbole_małe.count(ZNAK_KOMPUTER) == 2 and symbole_małe.count(" ") == 1: wynik -= 5

          # 5. Punkty za blokowanie wygranej przeciwnika na małej planszy
          if symbole_małe.count(ZNAK_KOMPUTER) == 2 and symbole_małe.count(ZNAK_GRACZ) == 1: wynik += 20
          if symbole_małe.count(ZNAK_GRACZ) == 2 and symbole_małe.count(ZNAK_KOMPUTER) == 1: wynik -= 20

  return wynik

In [43]:
def minimax_alpha_beta(plansza, duża_p, następna_p, depth, alpha, beta, maksymalizacja):
  # sprawdzanie, czy gra się skończyła lub czy osiągnęto limit głębokości
  wynik = sprawdź_wygrana_3x3(duża_p)
  if wynik == ZNAK_GRACZ: return 1000 + depth, None # wygrana komputera
  if wynik == ZNAK_KOMPUTER: return -1000 - depth, None   # wygrana gracza
  if depth == 0 or czy_remis(duża_p):
    return heurystyka(plansza, duża_p), None

  # ustalanie małych plansz, w których można grać
  if następna_p is None or duża_p[następna_p[0]][następna_p[1]] != " ":
    możliwe_sektory = [(w, k) for w in range(3) for k in range(3) if duża_p[w][k] == " "]
  else:
    możliwe_sektory = [następna_p]

  if maksymalizacja == True:
    najlepsza_wartość = -math.inf
    najlepszy_ruch = None

    for (d_w, d_k) in możliwe_sektory:
      for m_w in range(3):
        for m_k in range(3):
          if plansza[d_w][d_k][m_w][m_k] == " ":
            stary_stan = duża_p[d_w][d_k]
            plansza[d_w][d_k][m_w][m_k] = ZNAK_GRACZ
            wynik_tego_sektora = sprawdź_wygrana_3x3(plansza[d_w][d_k])
            if wynik_tego_sektora is None and czy_remis(plansza[d_w][d_k]) == True:
              wynik_tego_sektora = "R"
            if wynik_tego_sektora is not None:
              duża_p[d_w][d_k] = wynik_tego_sektora

            wynik_ruchu = minimax_alpha_beta(plansza, duża_p, (m_w, m_k), depth - 1, alpha, beta, False)[0]

            # cofnieęcie wartości pól
            plansza[d_w][d_k][m_w][m_k] = " "
            duża_p[d_w][d_k] = stary_stan

            if wynik_ruchu > najlepsza_wartość:
              najlepsza_wartość = wynik_ruchu
              najlepszy_ruch = (d_w, d_k, m_w, m_k)

            alpha = max(alpha, najlepsza_wartość)
            if beta <= alpha:
              break # odcięcie Alpha
        if beta <= alpha: break
      if beta <= alpha: break
    return najlepsza_wartość, najlepszy_ruch

  else:
    najlepsza_wartość = math.inf
    najlepszy_ruch = None

    for (d_w, d_k) in możliwe_sektory:
      for m_w in range(3):
        for m_k in range(3):
          if plansza[d_w][d_k][m_w][m_k] == " ":
            stary_stan = duża_p[d_w][d_k]
            plansza[d_w][d_k][m_w][m_k] = ZNAK_KOMPUTER
            wynik_tego_sektora = sprawdź_wygrana_3x3(plansza[d_w][d_k])
            if wynik_tego_sektora is None and czy_remis(plansza[d_w][d_k]):
              wynik_tego_sektora = "R"
            if wynik_tego_sektora is not None:
              duża_p[d_w][d_k] = wynik_tego_sektora

            wynik_ruchu = minimax_alpha_beta(plansza, duża_p, (m_w, m_k), depth - 1, alpha, beta, True)[0]

            plansza[d_w][d_k][m_w][m_k] = " "
            duża_p[d_w][d_k] = stary_stan

            if wynik_ruchu < najlepsza_wartość:
              najlepsza_wartość = wynik_ruchu
              najlepszy_ruch = (d_w, d_k, m_w, m_k)

            beta = min(beta, najlepsza_wartość)
            if beta <= alpha:
              break
        if beta <= alpha: break
      if beta <= alpha: break
    return najlepsza_wartość, najlepszy_ruch

MCTS (Monte Carlo Tree Search)

In [44]:
class StanGry:
  def __init__(self, plansza, duża_p, nastepna_p, tura):
    self.plansza = plansza
    self.duża_p = duża_p
    self.nastepna_p = nastepna_p
    self.tura = tura # 1 dla AI (maksymalizacja), -1 dla gracza

  # bierzemy możliwe ruchy do wykonania
  def weź_akcję(self):
    ruchy = []

    if self.nastepna_p is None or self.duża_p[self.nastepna_p[0]][self.nastepna_p[1]] != " ":
      sektory = [(w, k) for w in range(3) for k in range(3) if self.duża_p[w][k] == " "]
    else:
      sektory = [self.nastepna_p]

    for dw, dk in sektory:
      for mw in range(3):
        for mk in range(3):
          if self.plansza[dw][dk][mw][mk] == " ":
            ruchy.append((dw, dk, mw, mk))

    return ruchy

  def ruch(self, akcja):
    nowa_plansza = copy.deepcopy(self.plansza)
    nowa_duża = copy.deepcopy(self.duża_p)

    dw, dk, mw, mk = akcja
    znak = ZNAK_KOMPUTER if self.tura == 1 else ZNAK_GRACZ

    nowa_plansza[dw][dk][mw][mk] = znak
    stan_sektora(nowa_plansza, nowa_duża, dw, dk)

    if nowa_duża[mw][mk] == " ":
      nastepna_p = (mw, mk)
    else:
      nastepna_p = None

    return StanGry(nowa_plansza, nowa_duża, nastepna_p, -self.tura) # teraz dla przeciwnika

  def koniec_gry(self):
    if sprawdź_wygrana_3x3(self.duża_p) != None:
      return True
    if czy_remis(self.duża_p) == True:
      return True
    if not self.weź_akcję():
      return True
    return False


  def wynik_gry(self):
    wynik = sprawdź_wygrana_3x3(self.duża_p)
    if wynik == ZNAK_KOMPUTER: return 1
    if wynik == ZNAK_GRACZ: return -1
    return 0 # remis

In [45]:
class NodeDrzewa:
  def __init__(self, stan, rodzic=None, akcja_rodzica=None):
    self.stan = stan # stan planszy
    self.rodzic = rodzic
    self.akcja_rodzica = akcja_rodzica
    self.dzieci = [] # wszystkie możliwe kolejne ruchy
    self.liczba_wizyt = 0
    self.suma_wyników = 0  # suma wyników (-1, 0, 1)
    self.niewypróbowane_akcje = stan.weź_akcję() # wszytkie możliwe akcje

  # wybór
  def wartość_uct(self, c=1.41): # tzw. Upper Confidence Bound
    if self.liczba_wizyt == 0:
      return math.inf # zawsze odwiedzi nieodwiedzaną jeszcze drogę
    radzi_sobie = self.suma_wyników / self.liczba_wizyt # jak dobry ma wynik
    eksploracja = c * math.sqrt(math.log(max(1, self.rodzic.liczba_wizyt)) / self.liczba_wizyt) # aby odwiedzał nowe
    return radzi_sobie + eksploracja

  def rozrost(self):
    akcja = self.niewypróbowane_akcje.pop() # zabieranie i usuwanie elementu
    następny_stan = self.stan.ruch(akcja)
    node_dziecko = NodeDrzewa(następny_stan, rodzic=self, akcja_rodzica=akcja)
    self.dzieci.append(node_dziecko)
    return node_dziecko

  def symulacja(self):
    aktualny_stan = self.stan
    while not aktualny_stan.koniec_gry():
      możliwe_ruchy = aktualny_stan.weź_akcję()
      if not możliwe_ruchy:
        break
      akcja = random.choice(możliwe_ruchy)
      aktualny_stan = aktualny_stan.ruch(akcja)

    wynik_końcowy = aktualny_stan.wynik_gry()
    return wynik_końcowy * (-self.stan.tura)


  def propagacja_wsteczna(self, wynik):
    self.liczba_wizyt += 1
    self.suma_wyników += wynik
    if self.rodzic:
      self.rodzic.propagacja_wsteczna(-wynik)

  def najlepsze_dziecko(self):
    if not self.dzieci:
      return None

    zwycięzca = self.dzieci[0]  # na start zakładamy, że pierwsze dziecko jest najlepsze
    max_wizyt = zwycięzca.liczba_wizyt

    for dziecko in self.dzieci:
      if dziecko.liczba_wizyt > max_wizyt:
        max_wizyt = dziecko.liczba_wizyt
        zwycięzca = dziecko

    return zwycięzca


In [46]:
def mcts_ruch(plansza, duża_p, następna_p, iteracje=1000):
  stan_aktualny = StanGry(plansza, duża_p, następna_p, 1) # 1 = tura mcts, -1 = tura przeciwnik
  node_startowy = NodeDrzewa(stan_aktualny)

  for i in range(iteracje):
    # 1. wybór i rozrost
    node = node_startowy
    while not node.stan.koniec_gry():
      if len(node.niewypróbowane_akcje) > 0:
        node = node.rozrost()
        break
      else:
        najlepsze_dziecko = None
        najwyższe_uct = -math.inf

        for dziecko in node.dzieci:
          wartość = dziecko.wartość_uct()
          if wartość > najwyższe_uct:
            najwyższe_uct = wartość
            najlepsze_dziecko = dziecko

        node = najlepsze_dziecko

    # 2. symulacja
    wynik = node.symulacja()

    # 3. propagacja wsteczna
    node.propagacja_wsteczna(wynik)

  return node_startowy.najlepsze_dziecko().akcja_rodzica

GRA

In [47]:
plansza, duża_plansza = tworzenie_planszy()

while ZNAK_GRACZ != "X" and ZNAK_GRACZ != "O":
  ZNAK_GRACZ = input("Wybierz X lub O: ").upper()

ZNAK_KOMPUTER = "O" if ZNAK_GRACZ == "X" else "X"

while tura != "T" and tura != "N":
  tura = input("Czy chcesz zaczynać? (T/N): ").upper()

tura = "gracz" if tura == "T" else "komputer"

Wybierz X lub O: o
Czy chcesz zaczynać? (T/N): t


MINMAX VS MCTS

In [ ]:
wyświetlenie_planszy(plansza)

while koniec == None and remis == False:

  if tura == "gracz":
    print(f"\nRuch komputera_minmax: {ZNAK_GRACZ}.")
    ocena, ruch = minimax_alpha_beta(plansza, duża_plansza, następna_plansza, 6, -math.inf, math.inf, True)

    if ruch != None:
      pozycja_d_w, pozycja_d_k, pozycja_m_w, pozycja_m_k = ruch
      plansza[pozycja_d_w][pozycja_d_k][pozycja_m_w][pozycja_m_k] = ZNAK_GRACZ
      następna_plansza = (pozycja_m_w, pozycja_m_k)
      stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k)
      wypełnienie(plansza, duża_plansza)

      print(f"komputera_minmax zagrał na duża plansza: [{pozycja_d_w+1},{pozycja_d_k+1}], mała plansza:[{pozycja_m_w+1},{pozycja_m_k+1}]")
      wyświetlenie_planszy(plansza)
      time.sleep(0.5)
    else:
      print("Komputer_minmax nie znalazł ruchu!")

    koniec = sprawdź_wygrana_3x3(duża_plansza)
    remis = czy_remis(duża_plansza)

  else:
    print(f"\nRuch komputera_mcts: {ZNAK_KOMPUTER}.")
    ruch = mcts_ruch(plansza, duża_plansza, następna_plansza)

    if ruch != None:
      pozycja_d_w, pozycja_d_k, pozycja_m_w, pozycja_m_k = ruch
      plansza[pozycja_d_w][pozycja_d_k][pozycja_m_w][pozycja_m_k] = ZNAK_KOMPUTER
      następna_plansza = (pozycja_m_w, pozycja_m_k)
      stan_sektora(plansza, duża_plansza, pozycja_d_w, pozycja_d_k)
      wypełnienie(plansza, duża_plansza)

      print(f"Komputer_mcts zagrał na duża plansza: [{pozycja_d_w+1},{pozycja_d_k+1}], mała plansza:[{pozycja_m_w+1},{pozycja_m_k+1}]")
      wyświetlenie_planszy(plansza)
      time.sleep(0.5)
    else:
      print("Komputer_mcts nie znalazł ruchu!")

    koniec = sprawdź_wygrana_3x3(duża_plansza)
    remis = czy_remis(duża_plansza)

  tura = "komputer" if tura == "gracz" else "gracz"

if remis == True:
  print("Remis! Gra nierozstrzygnięta.")
else:
  if koniec == ZNAK_KOMPUTER:
    print(f"Wygrał komputer_mcts: {ZNAK_KOMPUTER}!")
  else:
    print(f"Wygrał komputer_minmax: {ZNAK_GRACZ}!")

-------------------------
|       |       |       |
|       |       |       |
|       |       |       |
-------------------------
|       |       |       |
|       |       |       |
|       |       |       |
-------------------------
|       |       |       |
|       |       |       |
|       |       |       |
-------------------------

Ruch komputera_minmax: O.
komputera_minmax zagrał na duża plansza: [1,1], mała plansza:[1,1]
-------------------------
| O     |       |       |
|       |       |       |
|       |       |       |
-------------------------
|       |       |       |
|       |       |       |
|       |       |       |
-------------------------
|       |       |       |
|       |       |       |
|       |       |       |
-------------------------

Ruch komputera_mcts: X.
Komputer_mcts zagrał na duża plansza: [1,1], mała plansza:[1,3]
-------------------------
| O   X |       |       |
|       |       |       |
|       |       |       |
-------------------------
|       |  